In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.7 MB/s eta 0:00:00:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [4]:
# model_name = "microsoft/deberta-v3-small"
# save_dir = "/kaggle/input/notebooks/ayanansari111/deberta-finetuned/deberta_finetuned"

# base_model = AutoModelForMultipleChoice.from_pretrained(model_name)
# tokenizer = AutoTokenizer.from_pretrained(save_dir)

# model = PeftModel.from_pretrained(base_model, save_dir)
# model.eval()

In [5]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType

train_subset = train.copy()

model_name = "microsoft/deberta-v3-small"
dtokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForMultipleChoice.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query_proj", "value_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
deberta = get_peft_model(base_model, lora_config)

raw_dataset = Dataset.from_pandas(train)

label_mapping = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
options_keys = ["A", "B", "C", "D", "E"]

def preprocess_function(examples):
    batch_input_ids = []
    batch_attention_mask = []
    
    for i in range(len(examples["prompt"])):
        prompt = examples["prompt"][i]
        formatted_opts = [str(prompt) + " [SEP] " + str(examples[opt][i]) for opt in options_keys]
        
        tokenized = dtokenizer(
            formatted_opts,
            padding="max_length",
            truncation=True,
            max_length=64 
        )
        batch_input_ids.append(tokenized["input_ids"])
        batch_attention_mask.append(tokenized["attention_mask"])
        
    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels": [label_mapping[ans] for ans in examples["answer"]]
    }

processed_dataset = raw_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=raw_dataset.column_names
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=5e-4,
    per_device_train_batch_size=4,       
    gradient_accumulation_steps=1,       
    max_steps=1000,                         
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=deberta,
    args=training_args,
    train_dataset=processed_dataset
)

train_output = trainer.train()

print(f"Global Step: {train_output.global_step}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

... (3 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
100,3.215020
200,2.922588
300,2.523457
400,2.185039
500,1.886255
600,1.692314
700,1.544829
800,1.404796
900,1.418823
1000,1.363534


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Global Step: 1000


# Q1

In [6]:
import torch.nn.functional as F
num = 25
first_item = processed_dataset[num]

model_inputs = {
    "input_ids": torch.tensor([first_item['input_ids']]).to(deberta.device),
    "attention_mask": torch.tensor([first_item['attention_mask']]).to(deberta.device)
}

deberta.eval()
with torch.no_grad():
    output = deberta(**model_inputs)
    
logits = output.logits
dprobs = F.softmax(logits,dim=-1)
print(dprobs)
print(train.iloc[num])

tensor([[0.0015, 0.0024, 0.0015, 0.0016, 0.9932]], device='cuda:0',
       dtype=torch.float16)
id                                                       26
prompt    Choose the correct answer: What is Hesse's pri...
A         Hesse's principle of transfer is a concept in ...
B         Hesse's principle of transfer is a concept in ...
C         Hesse's principle of transfer is a concept in ...
D         Hesse's principle of transfer is a concept in ...
E         Hesse's principle of transfer is a concept in ...
answer                                                    E
Name: 25, dtype: object


In [7]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType

train = train.copy()

model_name = "roberta-base"
rtokenizer = AutoTokenizer.from_pretrained(model_name)
rbase_model = AutoModelForMultipleChoice.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
roberta = get_peft_model(rbase_model, lora_config)

raw_dataset = Dataset.from_pandas(train)

label_mapping = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
options_keys = ["A", "B", "C", "D", "E"]

def preprocess_function(examples):
    batch_input_ids = []
    batch_attention_mask = []
    
    for i in range(len(examples["prompt"])):
        prompt = examples["prompt"][i]
        formatted_opts = [str(prompt) + " [SEP] " + str(examples[opt][i]) for opt in options_keys]
        
        tokenized = rtokenizer(
            formatted_opts,
            padding="max_length",
            truncation=True,
            max_length=64 
        )
        batch_input_ids.append(tokenized["input_ids"])
        batch_attention_mask.append(tokenized["attention_mask"])
        
    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels": [label_mapping[ans] for ans in examples["answer"]]
    }

rprocessed_dataset = raw_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=raw_dataset.column_names
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=5e-4,
    per_device_train_batch_size=4,       
    gradient_accumulation_steps=1,       
    max_steps=1000,                         
    logging_steps=100,
    report_to="none",
)

rtrainer = Trainer(
    model=roberta,
    args=training_args,
    train_dataset=rprocessed_dataset
)

train_output = rtrainer.train()

print(f"Global Step: {train_output.global_step}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

... (3 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
100,3.220324
200,2.899720
300,2.490574
400,2.043837
500,1.788236
600,1.718834
700,1.445412
800,1.253527
900,1.299668
1000,1.265185


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Global Step: 1000


In [8]:
import torch.nn.functional as F
num = 25
first_item = rprocessed_dataset[num]

model_inputs = {
    "input_ids": torch.tensor([first_item['input_ids']]).to(roberta.device),
    "attention_mask": torch.tensor([first_item['attention_mask']]).to(roberta.device)
}

roberta.eval()
with torch.no_grad():
    output = roberta(**model_inputs)
    
logits = output.logits
rprobs = F.softmax(logits,dim=-1)
print(rprobs)
print(train.iloc[num])

tensor([[0.0120, 0.0112, 0.0124, 0.0123, 0.9521]], device='cuda:0')
id                                                       26
prompt    Choose the correct answer: What is Hesse's pri...
A         Hesse's principle of transfer is a concept in ...
B         Hesse's principle of transfer is a concept in ...
C         Hesse's principle of transfer is a concept in ...
D         Hesse's principle of transfer is a concept in ...
E         Hesse's principle of transfer is a concept in ...
answer                                                    E
Name: 25, dtype: object


# Q2

In [12]:
avg_probs = (rprobs + dprobs)/2
avg_probs

tensor([[0.0068, 0.0068, 0.0069, 0.0070, 0.9726]], device='cuda:0')

# Q3

In [13]:
w_probs = (0.7*dprobs) + (0.3*rprobs)
w_probs

tensor([[0.0047, 0.0050, 0.0048, 0.0048, 0.9809]], device='cuda:0')

# Q4

In [14]:
import torch
pred_idx = torch.topk(w_probs,k=3).indices
pred_idx

tensor([[4, 1, 3]], device='cuda:0')

# Q6

In [19]:
train_subset = train.head(50)
raw_subset = Dataset.from_pandas(train_subset)

dprocessed_dataset = raw_subset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=raw_subset.column_names
)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [18]:
for i in range(50):
    

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 50
})